In [15]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef


In [16]:
def run_ml_pipeline(data_path):
    """
    Loads data, preprocesses it, and evaluates multiple classification models.

    Args:
        data_path (str): The file path for the merged dataset.
    """
    try:
        # Expand user path (e.g., '~') to get the absolute path
        data_path = os.path.expanduser(data_path)
        print(f"Loading data from: {data_path}")
        emb = pd.read_csv(data_path)
        print("Data loaded successfully.")
    except FileNotFoundError:
        print(f"Error: Data file not found at '{data_path}'. Please ensure the file exists.")
        return
    except Exception as e:
        print(f"An error occurred while loading the data: {e}")
        return

    # --- 1. Define Targets and Columns to Exclude from Features ---
    targets = ['n7fy23_recode', 'n7dy23_recode']
    
    # These are columns that are not predictive features (e.g., identifiers)
    # We also add the target columns to this list so they aren't used as predictors.
    cols_to_exclude_base = [
        'st_code', 'state23', 'dist23', 'Unnamed: 0', 'caseid', 'weight',
        'wave', 'state', 'district', 'state_district_survey',
        'district_shapefile_code', 'district_shapefile'
    ]
    # Combine base exclusions with targets and remove any duplicates
    cols_to_exclude = list(set(cols_to_exclude_base + targets))

    # --- 2. Dynamically Create Feature List ---
    # This creates the list of predictors by taking all columns in the dataframe
    # and removing the ones specified in the exclusion list.
    features = [col for col in emb.columns if col not in cols_to_exclude]
    print(f"\nUsing {len(features)} features as predictors.")
    # print("Predictor columns:", features) # Uncomment to see the full list

    # --- 3. Preprocessing ---
    # Identify categorical columns among the features to be encoded.
    categorical_cols = [col for col in features if emb[col].dtype == 'object']
    print(f"Encoding categorical columns: {categorical_cols}")

    # Encode categorical variables using LabelEncoder
    le = LabelEncoder()
    for col in categorical_cols:
        # Using .astype(str) to handle potential mixed types
        emb[col] = le.fit_transform(emb[col].astype(str))

    # Define feature matrix (X) and target matrix (y)
    X = emb[features]
    y = emb[targets]

    # --- 4. Model Evaluation ---
    def evaluate_model(X, y, model, target_name):
        """Splits data, imputes missing values, trains model, and returns performance metrics."""
        X_train, X_test, y_train, y_test = train_test_split(X, y[target_name], test_size=0.2, random_state=42)
        
        # Drop rows with missing target values for this specific split
        y_train = y_train.dropna()
        X_train = X_train.loc[y_train.index]
        
        y_test = y_test.dropna()
        X_test = X_test.loc[y_test.index]

        if len(y_test) == 0:
            print(f"Warning: No test samples left for target '{target_name}' after dropping NaNs. Skipping.")
            return 0, 0, 0, 0, 0, 0
            
        # *** FIX: Impute missing values in features ***
        # We use a SimpleImputer to fill NaN values with the mean of each column.
        # It's important to fit the imputer ONLY on the training data to avoid data leakage.
        imputer = SimpleImputer(strategy='mean')
        X_train = imputer.fit_transform(X_train)
        X_test = imputer.transform(X_test) # Use the same imputer to transform the test set

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        ca = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        mcc = matthews_corrcoef(y_test, y_pred)
        
        return auc, ca, f1, prec, recall, mcc

    # Define the models to be tested
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Random Forest': RandomForestClassifier(random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'Neural Network': MLPClassifier(random_state=42, max_iter=500),
        'SVM': SVC(probability=True, random_state=42)
    }

    # --- 5. Run Experiments and Print Results ---
    results = {}
    for target in targets:
        print(f"\n--- Evaluating models for target: {target} ---")
        results[target] = {}
        for model_name, model in models.items():
            print(f"Training {model_name}...")
            auc, ca, f1, prec, recall, mcc = evaluate_model(X, y, model, target)
            results[target][model_name] = [auc, ca, f1, prec, recall, mcc]

    # Print a summary of the results
    for target, model_results in results.items():
        print(f"\n--- Results for {target} ---")
        for model, metrics in model_results.items():
            print(f"{model}: AUC={metrics[0]:.4f}, CA={metrics[1]:.4f}, F1={metrics[2]:.4f}, Prec={metrics[3]:.4f}, Recall={metrics[4]:.4f}, MCC={metrics[5]:.4f}")

# --- How to use the function ---
if __name__ == '__main__':
    # **IMPORTANT**: This should be the path to the final merged data file
    # created by your previous script.
    merged_data_file = '~/SummerSchool_Team1_2025/data_son/merged_embstats_full_extract_main.csv'
    
    run_ml_pipeline(merged_data_file)


Loading data from: /home/jovyan/SummerSchool_Team1_2025/data_son/merged_embstats_full_extract_main.csv
Data loaded successfully.

Using 581 features as predictors.
Encoding categorical columns: ['urban', 'gender', 'age', 'caste']

--- Evaluating models for target: n7fy23_recode ---
Training Logistic Regression...


/cvmfs/cybergis.illinois.edu/software/conda/cybergisx/python3-0.9.4/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training Random Forest...
Training Gradient Boosting...
Training Neural Network...
Training SVM...

--- Evaluating models for target: n7dy23_recode ---
Training Logistic Regression...


/cvmfs/cybergis.illinois.edu/software/conda/cybergisx/python3-0.9.4/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training Random Forest...
Training Gradient Boosting...
Training Neural Network...
Training SVM...

--- Results for n7fy23_recode ---
Logistic Regression: AUC=0.6032, CA=0.5786, F1=0.5359, Prec=0.5708, Recall=0.5050, MCC=0.1535
Random Forest: AUC=0.5708, CA=0.5558, F1=0.5372, Prec=0.5394, Recall=0.5350, MCC=0.1102
Gradient Boosting: AUC=0.6227, CA=0.5869, F1=0.5539, Prec=0.5773, Recall=0.5322, MCC=0.1709
Neural Network: AUC=0.5723, CA=0.5611, F1=0.6429, Prec=0.5287, Recall=0.8202, MCC=0.1613
SVM: AUC=0.5836, CA=0.5177, F1=0.0090, Prec=0.4545, Recall=0.0045, MCC=-0.0038

--- Results for n7dy23_recode ---
Logistic Regression: AUC=0.6126, CA=0.6298, F1=0.7248, Prec=0.6481, Recall=0.8221, MCC=0.1953
Random Forest: AUC=0.5730, CA=0.5790, F1=0.6489, Prec=0.6419, Recall=0.6561, MCC=0.1234
Gradient Boosting: AUC=0.6200, CA=0.6219, F1=0.7149, Prec=0.6466, Recall=0.7993, MCC=0.1807
Neural Network: AUC=0.5450, CA=0.4967, F1=0.4090, Prec=0.6734, Recall=0.2937, MCC=0.0967
SVM: AUC=0.5871, CA=0.5965